In [26]:
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
load_dotenv()


True

In [27]:
file_path = 'deliveries.csv'
df = pd.read_csv(file_path)

print('Data Preview:')
display(df.head())

numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

if numeric_cols:
    tar_column = numeric_cols[0] 
    print(f"\nAuto-selected Column for Anomaly Detection: '{tar_column}'")
    
    mean_val = df[tar_column].mean()
    std_val = df[tar_column].std()
    
    anomalies = df[(df[tar_column] < mean_val - 2 * std_val) | (df[tar_column] > mean_val + 2 * std_val)]
else:
    print("\n==> Error: No numeric column found in this CSV file!")
    tar_column = None

Data Preview:


,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,2,2,NaN,NaN,NaN



Auto-selected Column for Anomaly Detection: 'match_id'


In [28]:
if tar_column and tar_column in df.columns:
    if not anomalies.empty:
        print(f"Anomalies Detected in '{tar_column}' Column!\n")
        display(anomalies.head(10)) 

        print("\nGenerating AI Summary via Gemini...")

        client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

        prompt = f"""
You are an expert data analyst. 
Analyze the following anomaly/outlier records detected in the '{tar_column}' column of a dataset:

{anomalies.head(20).to_string()}

Provide a concise 3-bullet-point executive summary explaining:
1. What unusual patterns or extreme values are occurring in this data.
2. What these key anomalies likely signify in the context of this dataset (e.g., performance spikes, operational issues, high risk, or data entry bugs).
3. 2 actionable insights or next steps based on these findings.
"""

        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
        )

        print("\n--- AI Data Analysis Summary ---")
        print(response.text)

    else:
        print(f"==> No Anomaly Detected in '{tar_column}'! Everything looks normal.")

Anomalies Detected in 'match_id' Column!



,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
164750,11137,1,Royal Challengers Bangalore,Chennai Super Kings,1,1,V Kohli,PA Patel,DL Chahar,0,...,0,0,0,0,1,0,1,NaN,NaN,NaN
164751,11137,1,Royal Challengers Bangalore,Chennai Super Kings,1,2,PA Patel,V Kohli,DL Chahar,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
164752,11137,1,Royal Challengers Bangalore,Chennai Super Kings,1,3,PA Patel,V Kohli,DL Chahar,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
164753,11137,1,Royal Challengers Bangalore,Chennai Super Kings,1,4,PA Patel,V Kohli,DL Chahar,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
164754,11137,1,Royal Challengers Bangalore,Chennai Super Kings,1,5,PA Patel,V Kohli,DL Chahar,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN
164755,11137,1,Royal Challengers Bangalore,Chennai Super Kings,1,6,PA Patel,V Kohli,DL Chahar,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
164756,11137,1,Royal Challengers Bangalore,Chennai Super Kings,2,1,V Kohli,PA Patel,Harbhajan Singh,0,...,0,0,0,0,1,0,1,NaN,NaN,NaN
164757,11137,1,Royal Challengers Bangalore,Chennai Super Kings,2,2,PA Patel,V Kohli,Harbhajan Singh,0,...,0,0,0,0,1,0,1,NaN,NaN,NaN
164758,11137,1,Royal Challengers Bangalore,Chennai Super Kings,2,3,V Kohli,PA Patel,Harbhajan Singh,0,...,0,0,0,0,1,0,1,NaN,NaN,NaN
164759,11137,1,Royal Challengers Bangalore,Chennai Super Kings,2,4,PA Patel,V Kohli,Harbhajan Singh,0,...,0,0,0,0,2,0,2,NaN,NaN,NaN



Generating AI Summary via Gemini...

--- AI Data Analysis Summary ---
Here's an executive summary of the detected anomalies:

*   **Unusual Pattern**: All identified "anomaly records" consistently point to a single `match_id`, 11137, representing detailed ball-by-ball events for one specific cricket match. The internal structure and sequence of events within these records (e.g., incrementing overs/balls, player actions) appear consistent and typical for a cricket match.
*   **Significance**: The anomaly likely stems from the `match_id` value (11137) itself being an outlier within the larger dataset's `match_id` distribution or intended scope. This could signify that the match belongs to a different data period (e.g., a newer season like 2019, if the dataset primarily covers older matches) or a source that deviates from the main dataset's expected range, rather than indicating data corruption within the match's play-by-play details.
*   **Actionable Insights/Next Steps**:
    1.  **Val